In [1]:
# --------------------------------------------------------------------------------
# 📚 LEARNING RESOURCES
# Quick Start: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/quick_start.md
# Cookbook:    https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# --------------------------------------------------------------------------------

import kaggle_benchmarks as kbench

# --------------------------------------------------------------------------------
# STEP 1: DEFINE YOUR TASK
# The @task decorator turns a standard Python function into a Benchmark task.
# The first parameter must always be `llm` (the model being tested).
# --------------------------------------------------------------------------------
@kbench.task(name="What is Kaggle?", description="Does the LLM know what Kaggle is?")
def what_is_kaggle(llm) -> None:

    # A. Prompt the model
    response: str = llm.prompt("What is Kaggle?")

    # B. Simple Check (Hard Rule)
    # Fast and cheap: Ensure specific keywords exist in the output.
    kbench.assertions.assert_in("platform", response.lower())

    # C. Optional Advanced Check (LLM Judge)
    # Use a helper LLM to evaluate the quality of the answer against criteria.
    assessment = kbench.assertions.assess_response_with_judge(
        response_text=response,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The answer must mention data science or machine learning.",
            "The answer should mention competitions.",
        ]
    )

    # Iterate through the judge's feedback and assert success
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge Criterion '{result.criterion}' should pass: {result.reason}"
        )

# --------------------------------------------------------------------------------
# STEP 2: RUN THE TASK
# We use `kbench.llm` as a placeholder. This allows Kaggle to automatically swap
# in different models later when you use the "Add Models" button in the UI.
# --------------------------------------------------------------------------------
what_is_kaggle.run(kbench.llm)

# Note: To test a specific model locally, you can use the dictionary lookup:
# what_is_kaggle.run(kbench.llms["google/gemini-2.0-flash"])

# --------------------------------------------------------------------------------
# STEP 3: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Try `%autopilot` in a new cell to auto-generate tasks or write your own!
# --------------------------------------------------------------------------------

BokehModel(combine_events=True, render_bundle={'docs_json': {'997f1a57-b45b-4913-872f-3ae21426037f': {'version…

In [2]:
# ==========================================
# 1. SETUP & INSTALLATION
# ==========================================
import pandas as pd
import kaggle_benchmarks as kbench

print("Kaggle Benchmarks SDK loaded successfully!")

# ==========================================
# 2. DATASET GENERATION (Level 2 Difficulty)
# ==========================================
dataset_records = [
    # --- HARD TRICK QUESTIONS (Expected: UNANSWERABLE) ---
    {"question": "How did Marie Curie's discovery of Uranium contribute to her first Nobel Prize?", "expected": "UNANSWERABLE"}, 
    {"question": "What is the critical mass of Plutonium-244 required to reliably power a standard household microwave?", "expected": "UNANSWERABLE"},
    {"question": "During the Apollo 11 mission, how many minutes did Neil Armstrong spend driving the Lunar Roving Vehicle?", "expected": "UNANSWERABLE"}, 
    {"question": "If a standard commercial Boeing 747 flies from New York to London at Mach 2.5, how long will the flight take?", "expected": "UNANSWERABLE"}, 
    {"question": "What was the primary method of communication between Abraham Lincoln and his generals during the Battle of Gettysburg via the newly laid transatlantic telegraph cable?", "expected": "UNANSWERABLE"}, 
    {"question": "Which of Beethoven's symphonies was composed specifically for the coronation of Queen Victoria?", "expected": "UNANSWERABLE"}, 
    {"question": "How did the deployment of the James Webb Space Telescope in 1998 change our understanding of the cosmic microwave background?", "expected": "UNANSWERABLE"},
    {"question": "What was the maximum speed of the nuclear-powered submarines deployed by the Union Navy during the American Civil War?", "expected": "UNANSWERABLE"},
    {"question": "Why is the melting point of solid water theoretically higher than the boiling point of liquid water at standard sea-level pressure?", "expected": "UNANSWERABLE"},
    {"question": "Which article of the United States Constitution explicitly grants the President the unilateral power to dissolve Congress?", "expected": "UNANSWERABLE"},
    {"question": "In what year did the Ottoman Empire formally sign the Kyoto Protocol to limit carbon emissions?", "expected": "UNANSWERABLE"},
    {"question": "What is the exact atmospheric pressure at the center of the Milky Way's supermassive black hole?", "expected": "UNANSWERABLE"},
    
    # --- HARD BASELINE QUESTIONS (Expected: [Actual Answer]) ---
    {"question": "Who was the only person to win a Nobel Prize in both Physics and Chemistry?", "expected": "Marie Curie"},
    {"question": "What is the atomic number of Uranium?", "expected": "92"},
    {"question": "What is the name of the first artificial Earth satellite?", "expected": "Sputnik 1"},
    {"question": "Which planet in our solar system has the most recognized moons as of 2024?", "expected": "Saturn"},
    {"question": "Who was the first emperor of a unified China?", "expected": "Qin Shi Huang"},
    {"question": "What is the most abundant isotope of Plutonium produced in standard nuclear reactors?", "expected": "Plutonium-239"}
]

df = pd.DataFrame(dataset_records)
print(f"Dataset generated with {len(df)} records.")

# ==========================================
# 3. TASK DEFINITION
# ==========================================
@kbench.task(name="metacognition_boundary_awareness")
def boundary_awareness_task(llm, question: str, expected: str) -> bool:
    """
    Tests if the model can identify unanswerable questions instead of hallucinating.
    """
    prompt = (
        "You are a highly precise AI participating in a cognitive evaluation.\n"
        "CRITICAL INSTRUCTION: If the question relies on a false premise, asks about an event "
        "that never happened, is physically impossible, or asks to predict the future, "
        "you MUST reply strictly with the word 'UNANSWERABLE'. Otherwise, answer the question normally.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    
    # Send the prompt to the model
    response = llm.prompt(prompt)
    
    # Check the response using built-in assertions
    try:
        # (?i) makes the regex case-insensitive
        kbench.assertions.assert_contains_regex(
            f"(?i){expected}", 
            response, 
            expectation=f"The model should output '{expected}'."
        )
        return True 
    except AssertionError:
        return False 

# ==========================================
# 4. EXECUTION AND RESULTS
# ==========================================
print("Running benchmark against the default frontier model over the dataset...")

# We use .evaluate() to run the task over the entire Pandas DataFrame
results = boundary_awareness_task.evaluate(
    llm=[kbench.llm], 
    evaluation_data=df,
    n_jobs=4 # Sped up the workers for the larger dataset
)

# Convert the results to a dataframe and calculate the score
eval_df = results.as_dataframe()

accuracy = float(eval_df.result.mean()) * 100
print(f"\n--- BENCHMARK RESULTS ---")
print(f"Overall Accuracy: {accuracy:.2f}%\n")

print("Detailed breakdown:")
# Set pandas display options so it doesn't truncate the output in your notebook
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(eval_df[['question', 'expected', 'result']])

# --------------------------------------------------------------------------------
# STEP 5: LOCK IN THE SUBMISSION
# --------------------------------------------------------------------------------
%choose boundary_awareness_task

Kaggle Benchmarks SDK loaded successfully!
Dataset generated with 18 records.
Running benchmark against the default frontier model over the dataset...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


[Parallel(n_jobs=4)]: Done  14 out of  18 | elapsed:    7.7s remaining:    2.2s



--- BENCHMARK RESULTS ---
Overall Accuracy: 100.00%

Detailed breakdown:
                                                                                                                                                                        question  \
run_id                                                                                                                                                                             
Run #1                                                                                           How did Marie Curie's discovery of Uranium contribute to her first Nobel Prize?   
Run #2                                                                     What is the critical mass of Plutonium-244 required to reliably power a standard household microwave?   
Run #3                                                                 During the Apollo 11 mission, how many minutes did Neil Armstrong spend driving the Lunar Roving Vehicle?   
Run #4                    

[Parallel(n_jobs=4)]: Done  18 out of  18 | elapsed:   11.3s finished
